# Stage 8 Explainer Notebook

This notebook mirrors Stage 8 scripts step-by-step:
- preflight checks
- topic runs
- lab runs
- optional Qdrant path
- deliverable verification


## 1) Environment and Helpers
Run this notebook from `red-book/src/stage-8`.

In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import json
import csv
from datetime import datetime

try:
    import pandas as pd
except Exception:
    pd = None

try:
    from IPython.display import display
except Exception:
    display = print

ROOT = Path.cwd()
print('Working dir:', ROOT)

def run_cmd(cmd):
    print(f'\n>>> {cmd}')
    p = subprocess.run(cmd, shell=True, cwd=ROOT, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if p.returncode != 0:
        raise RuntimeError(f'Command failed ({p.returncode}): {cmd}')

def run_py(script):
    run_cmd(f'{sys.executable} {script}')

def read_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows
    with path.open('r', encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                continue
    return rows

def read_csv_rows(path: Path):
    if not path.exists():
        return []
    with path.open('r', encoding='utf-8', errors='replace', newline='') as f:
        return list(csv.DictReader(f))


## 2) Preflight Checks

In [ ]:
run_cmd(f'{sys.executable} --version')
run_cmd(f'{sys.executable} -c "import numpy, sklearn; print(\"numpy/sklearn ready\")"')
run_cmd(f'{sys.executable} -c "import torch; print(\"torch\", torch.__version__, \"cuda_available\", torch.cuda.is_available())"')

## 3) Run Key Topic Scripts (Intermediate Path)

In [ ]:
# Toggle expert path execution
RUN_ADVANCED = False

topic_scripts = [
    'topic00_pytorch_cuda_tuning_intermediate.py',
    'topic01_dataset_quality_intermediate.py',
    'topic02_sft_intermediate.py',
    'topic03_lora_intermediate.py',
    'topic04_qlora_intermediate.py',
    'topic05_distill_intermediate.py',
    'topic06_tune_vs_rag_intermediate.py',
    'topic07_eval_regression_intermediate.py',
    'topic08_ops_observability_intermediate.py',
    'topic09_dpo_intermediate.py',
    'topic10_adapter_merge_intermediate.py',
    'topic11_synthetic_data_curation_intermediate.py',
    'topic12_memory_optimization_intermediate.py',
]

advanced_scripts = [
    'topic00c_pytorch_cuda_tuning_advanced.py',
    'topic01c_data_governance_advanced.py',
    'topic02c_sft_eval_advanced.py',
    'topic03c_lora_rank_sweep_advanced.py',
    'topic04c_qlora_memory_tradeoff_advanced.py',
    'topic05c_distill_eval_advanced.py',
    'topic06c_hybrid_strategy_advanced.py',
    'topic07c_promotion_gate_advanced.py',
    'topic08c_canary_rollback_advanced.py',
    'topic09c_dpo_eval_advanced.py',
    'topic10c_task_arithmetic_advanced.py',
    'topic11c_synthetic_data_governance_advanced.py',
    'topic12c_flashattn_checkpointing_advanced.py',
]

if RUN_ADVANCED:
    print('RUN_ADVANCED=True -> running intermediate + advanced topics')
    topic_scripts = topic_scripts + advanced_scripts
else:
    print('RUN_ADVANCED=False -> running intermediate topics only')

for s in topic_scripts:
    run_py(s)


## 4) Run Lab Set (Core)

In [ ]:
lab_scripts = [
    'lab01_instruction_tuning_baseline.py',
    'lab02_lora_qlora_comparison.py',
    'lab03_distillation_tradeoff_lab.py',
    'lab04_finetune_project_baseline_to_production.py',
]

for s in lab_scripts:
    run_py(s)

## 4.1) Hardware and VRAM Telemetry Review


In [ ]:
import torch

results = ROOT / 'results'
canon = results / 'stage8'
vram_csv = canon / 'vram_telemetry_5090.csv'

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    try:
        summary = torch.cuda.memory_summary(device=0, abbreviated=True)
        print('--- torch.cuda.memory_summary (abbrev, first 1200 chars) ---')
        print(summary[:1200])
    except Exception as e:
        print('memory_summary unavailable:', e)

if vram_csv.exists():
    print('\nLoaded telemetry:', vram_csv)
    if pd is not None:
        df = pd.read_csv(vram_csv)
        display(df)
        if 'peak_vram_mb' in df.columns:
            print('Peak VRAM MB (max row):', float(df['peak_vram_mb'].max()))
    else:
        print(vram_csv.read_text(encoding='utf-8', errors='replace')[:2000])
else:
    print('Telemetry file missing. Run lab02 first.')



## 4.2) Distillation Hard-Case Inspection (Teacher vs Student)


In [ ]:
teacher_path = results / 'lab3_teacher_outputs.jsonl'
student_path = results / 'lab3_student_outputs.jsonl'

teacher_rows = read_jsonl(teacher_path)
student_rows = read_jsonl(student_path)

if not teacher_rows or not student_rows:
    print('Distillation outputs missing. Run lab03 first.')
else:
    t_by_id = {r.get('id'): r for r in teacher_rows if r.get('id') is not None}
    s_by_id = {r.get('id'): r for r in student_rows if r.get('id') is not None}

    hard_cases = []
    for rid in sorted(set(t_by_id) & set(s_by_id)):
        t = t_by_id[rid]
        s = s_by_id[rid]
        t_pred = t.get('pred_label')
        s_pred = s.get('pred_label')
        t_reason = (((t.get('output') or {}).get('reason')) or '')
        s_reason = (((s.get('output') or {}).get('reason')) or '')
        if t_pred != s_pred or t_reason != s_reason:
            hard_cases.append({
                'id': rid,
                'gold': t.get('gold_label'),
                'teacher_pred': t_pred,
                'student_pred': s_pred,
                'teacher_reason': t_reason,
                'student_reason': s_reason,
                'input': t.get('input'),
            })

    print('Teacher rows:', len(teacher_rows), '| Student rows:', len(student_rows))
    print('Hard-case divergences found:', len(hard_cases))

    if hard_cases:
        sample = hard_cases[:5]
        if pd is not None:
            display(pd.DataFrame(sample))
        else:
            for row in sample:
                print(row)
    else:
        print('No divergence on this run; models behaved similarly on sampled cases.')

report_path = results / 'lab3_distillation_report.md'
if report_path.exists():
    print('\n--- Distillation report excerpt ---')
    print(report_path.read_text(encoding='utf-8', errors='replace')[:1500])



## 5) Optional: Run Qdrant Comparison Lab

In [ ]:
qdrant_url = 'http://localhost:6333/collections'

try:
    import urllib.request
    with urllib.request.urlopen(qdrant_url, timeout=1.0) as resp:
        ok = getattr(resp, 'status', 200) == 200
except Exception:
    ok = False

print('Qdrant available:', ok)
if ok:
    run_py('lab05_finetune_vs_rag_vs_hybrid_qdrant.py')
else:
    print('Skipping lab05 (Qdrant not reachable).')

## 5.1) Strategy Segmented Analysis (Fine-Tune vs RAG vs Hybrid)


In [ ]:
results = ROOT / 'results'
lab5 = results / 'lab5_compare_prompt_rag_tune.csv'

if lab5.exists():
    print('Using lab5 comparison file:', lab5)
    if pd is not None:
        df = pd.read_csv(lab5)
        display(df)
        cols = set(df.columns)
        if {'method', 'accuracy'}.issubset(cols):
            display(df.sort_values('accuracy', ascending=False))
    else:
        print(lab5.read_text(encoding='utf-8', errors='replace')[:2000])
else:
    print('lab5 comparison not found. Falling back to topic06 metrics for decision hints.')
    fallback_files = [
        results / 'topic06_tune_vs_rag_intermediate_metrics.csv',
        results / 'topic06c_hybrid_strategy_advanced_metrics.csv',
    ]
    rows = []
    for fp in fallback_files:
        rows.extend(read_csv_rows(fp))
    if not rows:
        print('No fallback strategy metrics found yet.')
    else:
        if pd is not None:
            df = pd.DataFrame(rows)
            wanted = ['topic_id', 'run_type', 'accuracy', 'f1_macro', 'format_validity', 'latency_ms', 'cost_per_1k_queries']
            cols = [c for c in wanted if c in df.columns]
            display(df[cols] if cols else df)
        else:
            for row in rows:
                print(row)

print('\nSegmented analysis checklist:')
print('- Prefer tuning for stable style/format control.')
print('- Prefer RAG for factual freshness and external source grounding.')
print('- Prefer hybrid when both style reliability and factual freshness are required.')



## 6) Verify Required Deliverables

In [ ]:
# Verify core outputs (labs 1-4)
run_cmd('powershell -ExecutionPolicy Bypass -File .\verify_stage8_outputs.ps1')

In [ ]:
# Verify with optional Qdrant outputs
if ok:
    run_cmd('powershell -ExecutionPolicy Bypass -File .\verify_stage8_outputs.ps1 -IncludeQdrant')
else:
    print('Qdrant verification skipped because local server is unavailable.')

## 6.1) Promotion Gate Logic (Notebook Simulation)


In [ ]:
results = ROOT / 'results'
metrics_path = results / 'lab4_metrics_comparison.csv'
forgetting_path = results / 'stage8' / 'forgetting_test_results.jsonl'

rows = read_csv_rows(metrics_path)
if not rows:
    print('lab4 metrics file missing. Run lab04 first.')
else:
    def val(row, *keys):
        for k in keys:
            if k in row and row[k] not in ('', None):
                try:
                    return float(row[k])
                except Exception:
                    return None
        return None

    by_metric = {r.get('metric'): r for r in rows if r.get('metric')}
    acc_delta = val(by_metric.get('accuracy', {}), 'delta') or 0.0
    fmt_base = val(by_metric.get('format_validity', {}), 'baseline_value', 'base_value')
    fmt_new = val(by_metric.get('format_validity', {}), 'improved_value', 'tuned_value')

    retention_pass = True
    frows = read_jsonl(forgetting_path)
    if frows:
        last = frows[-1]
        maybe = last.get('retention_gate_pass')
        if isinstance(maybe, bool):
            retention_pass = maybe

    promote = (acc_delta >= 0.0) and (fmt_new is not None and fmt_base is not None and fmt_new >= fmt_base) and retention_pass

    print('accuracy_delta =', acc_delta)
    print('format_validity baseline -> tuned =', fmt_base, '->', fmt_new)
    print('retention_gate_pass =', retention_pass)
    print('DECISION =', 'PROMOTE' if promote else 'HOLD/ROLLBACK')



## 6.2) Canary and Drift Evidence Preview


In [ ]:
results = ROOT / 'results'
canon = results / 'stage8'
drift_path = canon / 'hallucination_drift_log.jsonl'

if drift_path.exists():
    rows = read_jsonl(drift_path)
    print('Drift events:', len(rows))
    sample = rows[:10]
    if pd is not None and sample:
        display(pd.DataFrame(sample))
    else:
        for row in sample:
            print(row)
else:
    print('hallucination_drift_log.jsonl not generated yet (planned artifact).')
    fallback = results / 'topic08c_canary_rollback_advanced_metrics.csv'
    if fallback.exists():
        print('Fallback canary metrics:', fallback)
        if pd is not None:
            display(pd.read_csv(fallback))
        else:
            print(fallback.read_text(encoding='utf-8', errors='replace')[:1500])
    else:
        print('Run topic08c advanced script to generate canary baseline metrics.')



## 7) Inspect Output Files

In [ ]:
results = ROOT / 'results'
for p in sorted(results.glob('*')):
    size = p.stat().st_size
    print(f'{p.name:55} {size:>8} bytes')

print('\n--- Stage 8 Canonical Artifact Preview ---')
canon = results / 'stage8'
preview_targets = [
    canon / 'vram_telemetry_5090.csv',
    results / 'lab1_metrics_comparison.csv',
    results / 'lab2_memory_latency_report.md',
    results / 'lab3_distillation_report.md',
    canon / 'model_promotion_report.md',
    canon / 'hallucination_drift_log.jsonl',
    canon / 'dpo_preference_eval.csv',
    canon / 'adapter_merge_matrix.csv',
    canon / 'synthetic_data_curation_report.md',
    canon / 'flashattn_checkpointing_benchmark.csv',
]

for target in preview_targets:
    print(f'\n{target}')
    if not target.exists():
        print('not generated yet')
        continue

    if target.suffix.lower() == '.csv' and pd is not None:
        try:
            df = pd.read_csv(target)
            display(df.head(12))
            continue
        except Exception:
            pass

    if target.suffix.lower() == '.jsonl':
        rows = read_jsonl(target)
        if pd is not None and rows:
            display(pd.DataFrame(rows[:12]))
        else:
            print(target.read_text(encoding='utf-8', errors='replace')[:1200])
        continue

    text = target.read_text(encoding='utf-8', errors='replace')
    print(text[:1800])


## 8) Next-Step Reflection
Use generated reports to answer:
1. Which method won on quality?
2. Which method won on memory/latency?
3. What is your final promote/hold/rollback decision and why?